In [1]:
import torch
import torchvision
from torchvision import transforms, datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

ModuleNotFoundError: No module named 'torch'

In [ ]:
transform = transforms.Compose([
    transforms.ToTensor()
])

#Downloading the Training Data
print("Downloading Training Data...")
train_dataset = datasets.MNIST(root='./data',
                               train=True,
                               download=True,
                               transform=transform)

#Downloading the Testing Data
print("Downloading Testing Data...")
test_dataset = datasets.MNIST(root='./data',
                              train=False,
                              download=True,
                              transform=transform)

# Create DataLoaders
# This automatically groups your images into batches of 64.
# Shuffling the training data helps the network learn better.
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=True)
#Just to show that its working
print(f"Success! Loaded {len(train_dataset)} training images and {len(test_dataset)} testing images.")

In [ ]:
def imshow(img):
    npimg = img.numpy()
    plt.figure(figsize=(10, 2))
    plt.imshow(np.transpose(npimg, (1, 2, 0)), cmap='gray')
    plt.axis('off')
    plt.show()

# Grab one batch of the training images in the set
dataiter = iter(train_loader)
images, labels = next(dataiter)

# Show the first 100 images from the batch to see what the batch looks like
print("Sample MNIST Images:")
imshow(torchvision.utils.make_grid(images[:100]))
print("Labels for these images:", labels[:100].numpy())

In [ ]:
import torch.nn as nn

class Autoencoder(nn.Module):
    def __init__(self):
        super().__init__()

        # --- THE ENCODER ---
        # Squash the 784 pixels down to a 3-dimensional vector
        self.encoder = nn.Sequential(
            nn.Linear(28 * 28, 128),  # 784 -> 128
            nn.ReLU(),                # Activation function introduces non-linearity
            nn.Linear(128, 64),       # 128 -> 64
            nn.ReLU(),
            nn.Linear(64, 3)          # 64 -> 3
        )

        # --- THE DECODER ---
        # Expands the 3-dimensional vector back into 784 pixels
        self.decoder = nn.Sequential(
            nn.Linear(3, 64),         # 3 -> 64
            nn.ReLU(),
            nn.Linear(64, 128),       # 64 -> 128
            nn.ReLU(),
            nn.Linear(128, 28 * 28),  # 128 -> 784
            nn.Sigmoid()              # Sigmoid ensures output pixels are between 0 and 1(turns them to probablities)
        )

    def forward(self, x):
        #Flatten the incoming image batch from [64, 1, 28, 28] to [64, 784]
        x = x.view(-1, 28 * 28)

        # Pass through the encoder
        encoded = self.encoder(x)

        # Pass through the decoder
        decoded = self.decoder(encoded)

        # We return both of them so we can analyse the 'encoded' features later!
        return encoded, decoded

# Initialise the model to make sure it works
model = Autoencoder()
print(model)
#works

In [ ]:
import torch.optim as optim

# 1. Define the Loss Function and Optimiser
# MSE is used because we are comparing pixel values to pixel values.
criterion = nn.MSELoss()

# Adam is a fast, highly effective optimiser. (seen it in week 10 of lab)
optimiser = optim.Adam(model.parameters(), lr=0.001)

# Number of times we will pass the entire dataset through the network
epochs = 10

# We will store the average loss per epoch here to plot a graph
train_losses = []

print("Starting Training...")

for epoch in range(epochs):
    running_loss = 0.0

    # Iterate through the batches of 64 images from our train_loader
    for images, labels in train_loader:

        # Clear the old gradients from the last step
        optimiser.zero_grad()

        # Doing propagation
        # Our model returns both the encoded bottleneck and the decoded image.
        # We use '_' to ignore the encoded features for now, we only need 'decoded' to calculate loss.
        _, decoded = model(images)

        # Reshape the original images
        # The original images are shape [64, 1, 28, 28], but our decoded output is flattened at [64, 784].
        # We must flatten the original images so the shapes can be compared
        original_images_flattened = images.view(-1, 28 * 28)

        # Calculating the Loss
        loss = criterion(decoded, original_images_flattened)

        #Backward Pass: Calculate the gradients
        loss.backward()

        # F. Optimise: Tell the optimiser to update the network's weights based on the gradients
        optimiser.step()

        # Keep a running total of the loss
        running_loss += loss.item()

    # Calculate the average loss for this entire epoch
    epoch_loss = running_loss / len(train_loader)
    train_losses.append(epoch_loss)

    print(f"Epoch [{epoch+1}/{epochs}] - Loss: {epoch_loss:.4f}")



In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# 1. Put the model in evaluation mode
model.eval()

# 2. Turn off gradient calculations to speed up testing
with torch.no_grad():
    print("Model locked and ready for analysis.")


    # Grab just one single batch of images from the test set
    dataiter = iter(test_loader)
    images, labels = next(dataiter)

    # Pass the images through the model
    encoded_features, decoded_images = model(images)

    # Reshape the output back into 28x28 images so we can plot them
    decoded_images = decoded_images.view(-1, 1, 28, 28)

    # Plot the first 5 original images vs their reconstructions
    fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(10, 4))
    for i in range(5):
        # Top row: Originals
        axes[0, i].imshow(images[i].numpy().squeeze(), cmap='gray')
        axes[0, i].set_title("Original")
        axes[0, i].axis('off')

        # Bottom row: Reconstructions
        axes[1, i].imshow(decoded_images[i].numpy().squeeze(), cmap='gray')
        axes[1, i].set_title("Reconstructed")
        axes[1, i].axis('off')

    plt.tight_layout()
    plt.show()

In [ ]:

model.eval()
with torch.no_grad():
    # Grab a batch of images
    dataiter = iter(test_loader)
    images, labels = next(dataiter)

    # Pick two different images to morph between (e.g., index 0 and index 3)
    imgA = images[0].view(-1, 28 * 28)
    imgB = images[3].view(-1, 28 * 28)

    # 1. Encode both images down to their latent features
    latentA = model.encoder(imgA)
    latentB = model.encoder(imgB)

    # 2. Setting how many frames we want in our "morph" animation
    num_steps = 21
    fractions = torch.linspace(0, 1, num_steps)

    # 3. Create a plot with 15 subplots in a row
    fig, axes = plt.subplots(1, num_steps, figsize=(15, 2))

    for i, fraction in enumerate(fractions):
        # Blend the two latent vectors together step-by-step
        blended_latent = (1 - fraction) * latentA + fraction * latentB

        # Decode the new, blended vector back into a 28x28 image
        blended_image = model.decoder(blended_latent).view(28, 28)

        # Plot it
        axes[i].imshow(blended_image.numpy(), cmap='gray')
        axes[i].axis('off')

    plt.suptitle("Latent Space Interpolation: Morphing Features")
    plt.tight_layout()
    plt.show()
    # bbox_inches='tight' removes the extra white margins
# pad_inches=0.1 adds just a tiny sliver of padding so the title doesn't get cut off
    plt.savefig("morphing.png", bbox_inches='tight', pad_inches=0.1, dpi=300)

    plt.show()

In [ ]:

# Create the figure
plt.figure(figsize=(8, 5))

# Plot the training losses (Epochs on X-axis, Loss on Y-axis)
plt.plot(range(1, epochs + 1), train_losses, marker='o', linestyle='-', color='b', label='Training Loss')

plt.title('Autoencoder Training Loss Curve')
plt.xlabel('Epoch')
plt.ylabel('Mean Squared Error (MSE) Loss')

# Ensure the X-axis only shows whole numbers (1, 2, 3...)
plt.xticks(range(1, epochs + 1))

# Add a grid so it's easier to read the exact values
plt.grid(True, linestyle='--', alpha=0.7)
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
model.eval()
test_loss = 0.0

with torch.no_grad():
    for images, _ in test_loader:
        # Flatten the images to match our model's input
        images = images.view(-1, 28 * 28)

        # Forward pass (
        _, decoded = model(images)

        loss = criterion(decoded, images)

        test_loss += loss.item() * images.size(0)

# Calculate the average loss over the entire test set
avg_test_loss = test_loss / len(test_loader.dataset)

print(f"{avg_test_loss:.4f}")